In [1]:
import os
import getpass
import paramiko
import pandas as pd
import numpy as np

In [2]:
from smtplib import SMTP
from email.mime.text import MIMEText
from email.mime.multipart import MIMEMultipart
from email import encoders

In [3]:
def send_mail(body):

    message = MIMEMultipart()
    message['Subject'] = 'Python Isilon Quota Usage Report'
    message['From'] = 'operations@example.com'
    message['To'] = 'recipient@example.com'

    body_content = body
    #part1 = (MIMEText(body_content, "html"))
    message.attach(MIMEText(body_content, "html"))
    #message.attach(MIMEText(body_content, "plain"))
    #message.add(body_content)
    msg_body = message.as_string()
    #msg_body = body_content.as_string() #message.as_string()

    server = SMTP('smtp.example.com', 25)
    server.ehlo()
    #server.starttls()
    #server.login(message['From'], 'Unique password')
    server.sendmail(message['From'], message['To'], msg_body)
    server.quit()

In [4]:
ssh = paramiko.SSHClient()

In [6]:
cmd = "isi quota list --format csv --type directory"

In [7]:
u = "service-account@example.local"

In [8]:
p = getpass.getpass()

In [9]:
ssh.set_missing_host_key_policy(paramiko.WarningPolicy())
ssh.connect('storage.example.local', username=u, password=p)

In [10]:
del p
del u

In [11]:
stdin, stdout, stderr = ssh.exec_command(cmd)

In [12]:
cmdreturn = stdout.readlines()

In [13]:
ssh.close()

In [ ]:
cmdreturn

In [15]:
df = pd.DataFrame()

In [16]:
i=0
for line in cmdreturn:
    if i==0:
        i=1
        df = pd.DataFrame(columns=[line.split(',')])
    else:
       series = pd.Series(line.split(','), index=df.columns)
       df = df.append(series, ignore_index=True)

In [17]:
df = df.drop(['Type', 'AppliesTo', 'Snap', 'Adv', 'Hard', 'Soft'], axis=1)
df = df.iloc[:, :-1]

In [ ]:
df.head()

In [19]:
df2 = df
df2[['Used']] = df2[['Used']].astype(np.int64)
df2[['Used']] = df2[['Used']].divide(1024)
df2[['Used']] = df2[['Used']].divide(1024)
df2[['Used']] = df2[['Used']].divide(1024)
df2[['Used']] = df2[['Used']].astype(np.int64)
df2[['Path']] = df2[['Path']].astype(str)
df2.dtypes

Path    object
Used     int64
dtype: object

In [ ]:
df2.head()

In [21]:
df3 = df2

In [23]:
prefix = "/ifs/itisi1p/data/"
df3[['Path']] = df3[['Path']].astype(np.str_)
df3 = df3.replace(prefix,"",regex=True)

In [25]:
df3 = df3.rename(columns={'Path':'Tenant', 'Used':'Used GB'})

In [ ]:
df3

In [27]:
df4 = df3.to_html()

In [ ]:
print(df4)

In [29]:
send_mail(df4)